# Data Cleaning and Preprocessing Notebook

This notebook focuses only on cleaning and preprocessing the customer dataset.

In [1]:
import pandas as pd

file_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer.csv"
df = pd.read_csv(file_path)

df.head()

,customer_id,acquisition_date,acquisition_cost_usd,market_segment,supplier_id,order_id,order_date,order_value_usd,payment_date,satisfaction_score,support_tickets,lead_time_days
0,CUST-001,2024-01-05,850,North America,SUP-A,ORD-1001,2024-01-15,12500,2024-02-18,4,1,14
1,CUST-002,2024-01-06,920,Europe,SUP-B,ORD-1002,2024-01-18,8500,2024-02-25,5,0,16
2,CUST-003,2024-01-07,880,Asia-Pacific,SUP-C,ORD-1003,2024-01-20,21000,2024-03-01,4,2,15
3,CUST-004,2024-01-08,790,North America,SUP-A,ORD-1004,2024-01-22,7500,2024-02-28,3,1,13
4,CUST-005,2024-01-09,950,Europe,SUP-D,ORD-1005,2024-01-25,15000,2024-03-05,5,0,12


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (750, 12)

Columns:
['customer_id', 'acquisition_date', 'acquisition_cost_usd', 'market_segment', 'supplier_id', 'order_id', 'order_date', 'order_value_usd', 'payment_date', 'satisfaction_score', 'support_tickets', 'lead_time_days']

Missing values:
customer_id             0
acquisition_date        0
acquisition_cost_usd    0
market_segment          0
supplier_id             0
order_id                0
order_date              0
order_value_usd         0
payment_date            0
satisfaction_score      0
support_tickets         0
lead_time_days          0
dtype: int64

Duplicate rows: 0


In [3]:
# Remove obvious ID columns that usually do not help modeling
id_like_columns = ["customer_id", "order_id"]
existing_id_cols = [col for col in id_like_columns if col in df.columns]
df = df.drop(columns=existing_id_cols, errors="ignore")

print("Dropped ID columns:", existing_id_cols)

Dropped ID columns: ['customer_id', 'order_id']


In [ ]:
# Convert known date columns to datetime and engineer numeric date features
date_columns = ["acquisition_date", "order_date", "payment_date"]
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Derive interval-based features that are useful for prediction
if all(col in df.columns for col in ["acquisition_date", "order_date"]):
    df["days_acq_to_order"] = (df["order_date"] - df["acquisition_date"]).dt.days

if all(col in df.columns for col in ["order_date", "payment_date"]):
    df["days_order_to_payment"] = (df["payment_date"] - df["order_date"]).dt.days

# Calendar breakdown for each date column
for col in date_columns:
    if col in df.columns:
        df[f"{col}_year"] = df[col].dt.year
        df[f"{col}_month"] = df[col].dt.month
        df[f"{col}_dayofweek"] = df[col].dt.dayofweek

# Drop raw datetime columns so downstream models get compact numeric features
existing_date_cols = [col for col in date_columns if col in df.columns]
df = df.drop(columns=existing_date_cols, errors="ignore")

print("Date features created and raw date columns dropped.")
df.head()

,acquisition_date,order_date,payment_date
0,2024-01-05,2024-01-15,2024-02-18
1,2024-01-06,2024-01-18,2024-02-25
2,2024-01-07,2024-01-20,2024-03-01
3,2024-01-08,2024-01-22,2024-02-28
4,2024-01-09,2024-01-25,2024-03-05
...,...,...,...
745,2026-01-20,2029-04-02,2029-05-12
746,2026-01-21,2029-04-05,2029-05-15
747,2026-01-22,2029-04-08,2029-05-18
748,2026-01-23,2029-04-10,2029-05-20


In [5]:
# Fill missing values: median for numeric, mode for categorical
numeric_cols = df.select_dtypes(include=["number"]).columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining null values:")
print(df.isnull().sum())

Remaining null values:
acquisition_date        0
acquisition_cost_usd    0
market_segment          0
supplier_id             0
order_date              0
order_value_usd         0
payment_date            0
satisfaction_score      0
support_tickets         0
lead_time_days          0
dtype: int64


In [6]:
# Remove duplicate rows
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after = df.shape[0]

print(f"Rows before: {before}, after: {after}, removed: {before - after}")

Rows before: 750, after: 750, removed: 0


In [7]:
# Basic preprocessing: one-hot encode remaining categorical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
df_processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Processed shape:", df_processed.shape)
df_processed.head()

Processed shape: (750, 18)


,acquisition_date,acquisition_cost_usd,order_date,order_value_usd,payment_date,satisfaction_score,support_tickets,lead_time_days,market_segment_Asia-Pacific,market_segment_Europe,market_segment_Middle East,market_segment_North America,supplier_id_SUP-B,supplier_id_SUP-C,supplier_id_SUP-D,supplier_id_SUP-E,supplier_id_SUP-F,supplier_id_SUP-G
0,2024-01-05,850,2024-01-15,12500,2024-02-18,4,1,14,False,False,False,True,False,False,False,False,False,False
1,2024-01-06,920,2024-01-18,8500,2024-02-25,5,0,16,False,True,False,False,True,False,False,False,False,False
2,2024-01-07,880,2024-01-20,21000,2024-03-01,4,2,15,True,False,False,False,False,True,False,False,False,False
3,2024-01-08,790,2024-01-22,7500,2024-02-28,3,1,13,False,False,False,True,False,False,False,False,False,False
4,2024-01-09,950,2024-01-25,15000,2024-03-05,5,0,12,False,True,False,False,False,False,True,False,False,False


In [8]:
output_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv"
df_processed.to_csv(output_path, index=False)
print("Saved cleaned data to:", output_path)

Saved cleaned data to: /home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv
